# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24pwai0015-max/flyrank-ml-muhammad-arsalan/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This contract specifies the schema, grain, time windows, field classifications, and data limitations for the 30,000-page content refresh dataset (`content_refresh_anonymized.csv`). Every claim is verified with executable Python/pandas queries below.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis (Grain):**
**One row = one unique content page** identified by pseudonymous `content_id` (30,000 unique rows), belonging to one of 32 distinct clients (`client_id`). Every row represents a single page with at least 1 impression in the 90-day window and a content age of at least 90 days.

**Time Windows:**
- **Overall Observation Window:** Trailing 90 calendar days ending at export snapshot time (`impressions_90d`, `sessions_90d`, `clicks_90d`, etc.).
- **Recent Comparison Window:** Most recent 30 days (`*_last_30d`: days 1–30 back).
- **Prior Comparison Window:** Preceding 30 days (`*_prev_30d`: days 31–60 back).
- **Trend Outcome Window:** The delta between `last_30d` and `prev_30d` generates `trend_direction` and `trend_pct`.

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# Colab setup check
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/24pwai0015-max/flyrank-ml-muhammad-arsalan'
REPO_DIR = 'flyrank-ml-muhammad-arsalan'
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

csv_path = 'data/raw/content_refresh_anonymized.csv'
assert os.path.exists(csv_path), f'CSV not found at {csv_path}'
df = pd.read_csv(csv_path)

# Verify Grain
total_rows = len(df)
unique_content = df['content_id'].nunique()
unique_clients = df['client_id'].nunique()
duplicate_content_ids = df['content_id'].duplicated().sum()

print(f'Total Rows in Dataset: {total_rows:,}')
print(f'Unique content_id: {unique_content:,} (Duplicates: {duplicate_content_ids})')
print(f'Unique client_id: {unique_clients}')
print(f'Min content_age_days: {df["content_age_days"].min()} (all >= 90 days)')
print(f'Min impressions_90d: {df["impressions_90d"].min()} (all >= 1 impression)')
assert total_rows == unique_content, 'Grain violation: content_id is not unique per row'

Total Rows in Dataset: 30,000
Unique content_id: 30,000 (Duplicates: 0)
Unique client_id: 32
Min content_age_days: 90 (all >= 90 days)
Min impressions_90d: 1 (all >= 1 impression)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

All 44 columns in the dataset are classified into exactly one of four buckets:

| Bucket | Columns | Description / Rationale |
|---|---|---|
| **Features (Numeric & Categorical)** | `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier` | Known historical and content signals prior to or over the 90-day window. Safe to use for modeling. |
| **Label / Target** | `is_declining_label` (derived from `trend_direction == 'down'`) | The target decision variable: whether page impressions dropped > 20% between prev-30d and last-30d. |
| **Context** | `content_id`, `client_id` | Identifiers used exclusively for joining, indexing, and grouped client-holdout cross-validation splits. Never used as training features. |
| **Excluded** | `trend_direction`, `trend_pct`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `provider_used`, `model_used`, `char_count_tier`, `age_tier_order` | **Why Excluded:**<br>• `trend_direction`, `trend_pct`, `*_last_30d`, `*_prev_30d`: **Direct label leakage** (they mathematically constitute the target).<br>• `provider_used`, `model_used`: 71.5% missing, internal generation metadata, not generalizable.<br>• `char_count_tier`, `age_tier_order`: Redundant with numeric `word_count` / `content_age_days`. |

In [2]:
# Field bucket verification
features = [
    'search_volume', 'competition', 'competition_level', 'cpc', 'content_type',
    'main_intent', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'ai_traffic_pct', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]
context_cols = ['content_id', 'client_id']
label_source_cols = ['trend_direction', 'trend_pct']
excluded_cols = [
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'provider_used', 'model_used', 'char_count_tier', 'age_tier_order'
]

all_classified = set(features + context_cols + label_source_cols + excluded_cols)
dataset_cols = set(df.columns)

print(f'Total columns in dataset: {len(dataset_cols)}')
print(f'Total columns classified: {len(all_classified)}')
print(f'Unclassified columns: {dataset_cols - all_classified}')
assert dataset_cols == all_classified, 'Mismatch in column classification!'

Total columns in dataset: 44
Total columns classified: 44
Unclassified columns: set()


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Below we run systematic verification checks on:
1. **Row counts per client** (distribution across the 32 clients).
2. **Target distribution** (`trend_direction` counts and `is_declining_label` base rate).
3. **Missingness analysis** (identifying systematic gaps like keyword data missing in specific content types).
4. **Derived rate bounds & zero position flags** (validating that `avg_position == 0` is missingness, not rank 0).

In [3]:
# 1. Rows per Client Distribution (Top 10 & Summary)
client_counts = df['client_id'].value_counts()
print('=== CLIENT DISTRIBUTION SUMMARY ===')
print(f'Min pages per client: {client_counts.min():,}')
print(f'Median pages per client: {client_counts.median():,}')
print(f'Max pages per client: {client_counts.max():,}')
print('\nTop 5 Clients by Volume:')
print(client_counts.head(5))

# 2. Target Label Distribution
print('\n=== TREND DIRECTION & TARGET DISTRIBUTION ===')
trend_counts = df['trend_direction'].value_counts()
print(trend_counts)
declining_rate = (df['trend_direction'] == 'down').mean()
print(f'\nTarget Base Rate (is_declining_label = 1): {declining_rate:.4f} ({declining_rate:.1%})')

# 3. Missing Value Analysis by Column
missing_series = df.isnull().sum()
missing_table = pd.DataFrame({
    'missing_count': missing_series[missing_series > 0],
    'missing_pct': (missing_series[missing_series > 0] / len(df) * 100).round(2)
}).sort_values('missing_count', ascending=False)
print('\n=== MISSINGNESS AUDIT ===')
print(missing_table)

# 4. Systematic Missingness by Content Type
print('\n=== SYSTEMATIC MISSINGNESS: Keyword Metrics by Content Type ===')
missing_by_type = df.groupby('content_type')[['search_volume', 'competition', 'cpc', 'word_count']].apply(
    lambda g: g.isnull().mean() * 100
).round(2)
print(missing_by_type)

# 5. Check avg_position == 0 flag (No position data indicator)
zero_pos_count = (df['avg_position'] == 0).sum()
print(f'\nPages with avg_position == 0 (no position data): {zero_pos_count:,} ({zero_pos_count/len(df):.2%})')

=== CLIENT DISTRIBUTION SUMMARY ===
Min pages per client: 3
Median pages per client: 567.0
Max pages per client: 7,008

Top 5 Clients by Volume:
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
Name: count, dtype: int64

=== TREND DIRECTION & TARGET DISTRIBUTION ===
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Target Base Rate (is_declining_label = 1): 0.5421 (54.2%)

=== MISSINGNESS AUDIT ===
                   missing_count  missing_pct
provider_used              21438        71.46
word_count_tier             7699        25.66
char_count                  7699        25.66
word_count                  7699        25.66
char_count_tier             7699        25.66
model_used                  5733        19.11
trend_pct                   3388        11.29
competition_level           2610         8.70
search_volume       

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data CANNOT tell you (Honest Constraints):**

1. **Aggregated 90-Day Slice (No Daily Trajectory):** The dataset provides total sums over 90 days. It cannot reveal intra-window volatility, sudden algorithm updates on day 45, or multi-week seasonality patterns.
2. **Causal Reasons for Decline:** `is_declining_label` measures an observed drop (>20% impression loss), but cannot isolate the root cause (SERP feature changes, competitor launches, seasonal dips, or cannibalization).
3. **Zero-Impression Censoring:** Every page in this slice has `impressions_90d >= 1` and `content_age_days >= 90`. Zero-traffic legacy pages and newly published pages (<90 days old) are excluded.
4. **Systematic Gaps in Keyword Metadata:** Certain formats (like `feedly article`) never have search volume or CPC estimates; filling blanks with 0 must be handled carefully so models don't falsely equate 'missing keyword data' with 'zero search interest'.

In [4]:
# Data limits verification
print('=== VERIFYING DATA LIMITS & BOUNDS ===')
print(f'1. Minimum impressions_90d: {df["impressions_90d"].min()} (Zero-traffic pages excluded)')
print(f'2. Minimum content_age_days: {df["content_age_days"].min()} (Pages <90 days excluded)')
print(f'3. Feed articles missing search_volume: {df[df["content_type"] == "feedly article"]["search_volume"].isnull().mean():.1%}')
print(f'4. Ratio of declining vs non-declining: {df["trend_direction"].eq("down").sum()} down vs {df["trend_direction"].ne("down").sum()} other')
print('\nContract verified successfully across all 30,000 rows.')

=== VERIFYING DATA LIMITS & BOUNDS ===
1. Minimum impressions_90d: 1 (Zero-traffic pages excluded)
2. Minimum content_age_days: 90 (Pages <90 days excluded)
3. Feed articles missing search_volume: 100.0%
4. Ratio of declining vs non-declining: 16262 down vs 13738 other

Contract verified successfully across all 30,000 rows.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.